# Retreino Ponderado do BCS (correção de desbalanceamento)

Continuação de `01_treino_bcs.ipynb`. O modelo original mostrou erro
consistentemente maior nas classes extremas (3.25 e 4.25), com a
predição "comprimida" em direção à classe majoritária (3.75). Este
notebook testa duas estratégias de correção via peso por classe, e
documenta o resultado de cada uma — incluindo o resultado de que o
efeito foi real, porém limitado.

Pressupõe que as células de Setup/Configuração/Restauração de dataset/
Manifest/Split/Pipeline de `01_treino_bcs.ipynb` já rodaram nesta sessão
(reproduzidas de forma resumida abaixo, para o notebook ser executável
de forma independente).

## 1. Setup e reconstrução do estado

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:

!pip install -q scikit-learn pandas matplotlib pillow


In [ ]:
import os
import re

DATASET_ROOT = '/content/dataset_BCS_local'  
OUTPUT_DIR = '/content/drive/MyDrive/bcs_training_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MANIFEST_PATH = os.path.join(OUTPUT_DIR, 'manifest.csv')
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'best_model.keras')
TFLITE_OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'bcs-model-float32.tflite')
HISTORY_PLOT_PATH = os.path.join(OUTPUT_DIR, 'training_history.png')

BCS_CLASSES = ['3.25', '3.5', '3.75', '4.0', '4.25']
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

FILENAME_PATTERN = re.compile(r'^([A-Za-z]+_\d+)_(\d+)\.(jpg|jpeg|png)$', re.IGNORECASE)
ALT_PATTERN = re.compile(r'^([A-Za-z]+)-i(\d+)\.(jpg|jpeg|png)$', re.IGNORECASE)

In [ ]:
!cp /content/drive/MyDrive/dataset_BCS_local.zip /content/
!cd /content && unzip -q dataset_BCS_local.zip

Carrega o manifest já construído em `01_treino_bcs.ipynb` (evita reprocessar).

In [ ]:
import pandas as pd

manifest = pd.read_csv(MANIFEST_PATH)
print(f'{len(manifest)} imagens no manifest.')


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def group_split(df, group_col, test_size, seed):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx_a, idx_b = next(splitter.split(df, groups=df[group_col]))
    return df.iloc[idx_a].reset_index(drop=True), df.iloc[idx_b].reset_index(drop=True)


reliable_df = manifest[manifest['reliable_group']].reset_index(drop=True)
unreliable_df = manifest[~manifest['reliable_group']].reset_index(drop=True)

print(f"Imagens confiáveis (serão divididas entre treino/val/teste): {len(reliable_df)}")
print(f"Imagens não-confiáveis (vão inteiras para o treino): {len(unreliable_df)}")
print()

# Split (70/15/15) só acontece na parte confiável
train_reliable, temp_df = group_split(reliable_df, 'group', test_size=0.30, seed=SEED)
val_df, test_df = group_split(temp_df, 'group', test_size=0.50, seed=SEED)

# Treino final = parte confiável do treino + TODAS as imagens não-confiáveis
train_df = pd.concat([train_reliable, unreliable_df], ignore_index=True)

# check crítico: nenhuma vaca confiável pode aparecer em mais de um split
train_groups = set(train_reliable['group'])
val_groups = set(val_df['group'])
test_groups = set(test_df['group'])

assert not (train_groups & val_groups), 'VAZAMENTO: vaca em treino E validação!'
assert not (train_groups & test_groups), 'VAZAMENTO: vaca em treino E teste!'
assert not (val_groups & test_groups), 'VAZAMENTO: vaca em validação E teste!'
print('OK — nenhuma vaca confiável aparece em mais de um split.')
print()

for name, split_df in [('Treino', train_df), ('Validação', val_df), ('Teste', test_df)]:
    n_groups = split_df['group'].nunique()
    print(f'{name}: {len(split_df)} imagens, {n_groups} grupos')
    print(split_df['bcs'].value_counts().sort_index().to_dict())
    print()

In [ ]:
import tensorflow as tf

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label


def make_dataset(df, shuffle, batch_size=BATCH_SIZE):
    filepaths = df['filepath'].values
    labels = df['bcs'].values.astype('float32')

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)


## 2. Pesos por classe

Peso inversamente proporcional à frequência — classes raras (3.25, 4.25)
recebem peso maior na loss.

In [ ]:
class_counts = train_df['bcs'].value_counts()
class_weights_map = (class_counts.sum() / class_counts).to_dict()

print("Peso por classe (quanto maior, mais rara/mais peso na loss):")
for k, v in sorted(class_weights_map.items()):
    print(f"  {k}: {v:.3f}")

sample_weights = train_df['bcs'].map(class_weights_map).values.astype('float32')

Peso por classe (quanto maior, mais rara/mais peso na loss):
  3.25: 7.153
  3.5: 4.007
  3.75: 3.788
  4.0: 4.280
  4.25: 8.848


## 3. Dataset de treino ponderado

In [ ]:
def make_weighted_dataset(df, weights, shuffle, batch_size=BATCH_SIZE):
    filepaths = df['filepath'].values
    labels = df['bcs'].values.astype('float32')

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels, weights))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)

    def load_with_weight(filepath, label, weight):
        image, label = load_image(filepath, label)
        return image, label, weight

    ds = ds.map(load_with_weight, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_weighted = make_weighted_dataset(train_df, sample_weights, shuffle=True)

In [ ]:
val_ds = make_dataset(val_df, shuffle=False)


## 4. Definição do modelo (mesma arquitetura do notebook 1)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
    layers.RandomZoom(0.1),
], name='data_augmentation')


def build_model(img_size, trainable_base=False):
    base_model = keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3), include_top=False, weights='imagenet',
    )
    base_model.trainable = trainable_base

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = data_augmentation(inputs)
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=trainable_base)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='linear', name='bcs_output')(x)

    return keras.Model(inputs, outputs, name='bcs_regressor'), base_model


In [ ]:
CHECKPOINT_PATH_V2 = os.path.join(OUTPUT_DIR, 'best_model_weighted.keras')
TFLITE_OUTPUT_PATH_V2 = os.path.join(OUTPUT_DIR, 'bcs-model-weighted-float32.tflite')

## 5. Experimento — Huber loss + peso linear por classe

In [ ]:
model_v2, base_model_v2 = build_model(IMG_SIZE, trainable_base=False)
model_v2.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "bcs_regressor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bcs_output (Dense)              │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

Total params: 2,340,033 (8.93 MB)

Trainable params: 82,049 (320.50 KB)

Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.Huber(),
    metrics=['mae'],
)

callbacks_phase1_v2 = [
    keras.callbacks.EarlyStopping(monitor='val_mae', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=4, min_lr=1e-6),
]

history_phase1_v2 = model_v2.fit(
    train_ds_weighted,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_phase1_v2,
)

Epoch 1/20
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 174s 133ms/step - loss: 1.2148 - mae: 0.5550 - val_loss: 0.0719 - val_mae: 0.3053 - learning_rate: 0.0010
Epoch 2/20
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 141s 113ms/step - loss: 0.7822 - mae: 0.4375 - val_loss: 0.0940 - val_mae: 0.3568 - learning_rate: 0.0010
Epoch 3/20
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 138s 110ms/step - loss: 0.6134 - mae: 0.3816 - val_loss: 0.0524 - val_mae: 0.2623 - learning_rate: 0.0010
Epoch 4/20
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 138s 110ms/step - loss: 0.4830 - mae: 0.3341 - val_loss: 0.0455 - val_mae: 0.2443 - learning_rate: 0.0010
Epoch 5/20
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 140s 112ms/step - loss: 0.3999 - mae: 0.3015 - val_loss: 0.0470 - val_mae: 0.2475 - learning_rate: 0.0010
Epoch 6/20
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 137s 109ms/step - loss: 0.3399 - mae: 0.2764 - val_loss: 0.0474 - val_mae: 0.2483 - learning_rate: 0.0010
Epoch 7/20
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 145s 112ms/step - loss: 0.2977 - mae: 0.2572 - val_loss: 0.0405 - val_

**Nota sobre a Fase 2 (fine-tuning):** a sessão do Colab desconectou no
meio da época 12 de 15. Como o `ModelCheckpoint` salva a melhor versão a
cada época, o melhor resultado (época 11, `val_mae: 0.1967`) já estava
salvo no Drive — não foi necessário retreinar do zero. O log abaixo
mostra as 11 épocas que de fato completaram.

In [ ]:
base_model_v2.trainable = True
fine_tune_at = int(len(base_model_v2.layers) * 0.7)
for layer in base_model_v2.layers[:fine_tune_at]:
    layer.trainable = False

model_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.Huber(),
    metrics=['mae'],
)

callbacks_phase2_v2 = [
    keras.callbacks.EarlyStopping(monitor='val_mae', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=4, min_lr=1e-7),
    keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH_V2, monitor='val_mae', save_best_only=True),
]

history_phase2_v2 = model_v2.fit(
    train_ds_weighted,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_phase2_v2,
)

Epoch 1/15
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 200s 147ms/step - loss: 0.2291 - mae: 0.2227 - val_loss: 0.0348 - val_mae: 0.2108 - learning_rate: 1.0000e-05
Epoch 2/15
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 164s 131ms/step - loss: 0.1860 - mae: 0.2024 - val_loss: 0.0328 - val_mae: 0.2042 - learning_rate: 1.0000e-05
Epoch 3/15
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 163s 130ms/step - loss: 0.1662 - mae: 0.1915 - val_loss: 0.0323 - val_mae: 0.2023 - learning_rate: 1.0000e-05
Epoch 4/15
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 206s 134ms/step - loss: 0.1518 - mae: 0.1842 - val_loss: 0.0317 - val_mae: 0.2005 - learning_rate: 1.0000e-05
Epoch 5/15
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 219s 147ms/step - loss: 0.1412 - mae: 0.1781 - val_loss: 0.0308 - val_mae: 0.1977 - learning_rate: 1.0000e-05
Epoch 6/15
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 183s 132ms/step - loss: 0.1301 - mae: 0.1714 - val_loss: 0.0315 - val_mae: 0.1997 - learning_rate: 1.0000e-05
Epoch 7/15
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 200s 130ms/step - loss: 0.1219 - mae: 0.1666 -

## 6. Avaliação


In [ ]:
model_v2 = keras.models.load_model(CHECKPOINT_PATH_V2)

y_pred_v2 = model_v2.predict(test_ds).flatten()
y_true = test_df['bcs'].values

mae_v2 = np.mean(np.abs(y_true - y_pred_v2))
within_exact_v2 = np.mean(np.abs(y_true - y_pred_v2) <= 0.125) * 100
within_one_class_v2 = np.mean(np.abs(y_true - y_pred_v2) <= 0.25) * 100

print(f'MAE no teste (v2, ponderado): {mae_v2:.3f}')
print(f'Acerto exato (±0.125): {within_exact_v2:.1f}%')
print(f'Acerto com tolerância de 1 classe (±0.25): {within_one_class_v2:.1f}%')


MAE no teste (v2, ponderado): 0.206
Acerto exato (±0.125): 35.7%
Acerto com tolerância de 1 classe (±0.25): 67.0%


In [ ]:
test_df_eval_v2 = test_df.copy()
test_df_eval_v2['pred'] = y_pred_v2
test_df_eval_v2['abs_error'] = np.abs(test_df_eval_v2['bcs'] - test_df_eval_v2['pred'])
print(test_df_eval_v2.groupby('bcs')['abs_error'].mean())


bcs
3.25    0.305665
3.50    0.152010
3.75    0.154021
4.00    0.199674
4.25    0.324868
Name: abs_error, dtype: float64


## 7. Experimento — MSE + peso quadrático (mais agressivo)

Testa se um peso mais forte, com uma loss que não amortece erro grande
(diferente de Huber), reduz mais o viés nos extremos.

In [ ]:
class_counts = train_df['bcs'].value_counts()
class_weights_map_v3 = ((class_counts.sum() / class_counts) ** 2).to_dict()

sample_weights_v3 = train_df['bcs'].map(class_weights_map_v3).values.astype('float32')
train_ds_weighted_v3 = make_weighted_dataset(train_df, sample_weights_v3, shuffle=True)

CHECKPOINT_PATH_V3 = os.path.join(OUTPUT_DIR, 'best_model_v3.keras')

model_v3, base_model_v3 = build_model(IMG_SIZE, trainable_base=False)
model_v3.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])

history_phase1_v3 = model_v3.fit(
    train_ds_weighted_v3, validation_data=val_ds, epochs=20,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_mae', patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=4, min_lr=1e-6),
    ],
)

base_model_v3.trainable = True
fine_tune_at = int(len(base_model_v3.layers) * 0.7)
for layer in base_model_v3.layers[:fine_tune_at]:
    layer.trainable = False

model_v3.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5), loss='mse', metrics=['mae'])

history_phase2_v3 = model_v3.fit(
    train_ds_weighted_v3, validation_data=val_ds, epochs=15,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_mae', patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=4, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH_V3, monitor='val_mae', save_best_only=True),
    ],
)


## 8. Avaliação (resultado registrado)

In [ ]:
y_pred_v3 = model_v3.predict(test_ds).flatten()

mae_v3 = np.mean(np.abs(y_true - y_pred_v3))
within_exact_v3 = np.mean(np.abs(y_true - y_pred_v3) <= 0.125) * 100
within_one_class_v3 = np.mean(np.abs(y_true - y_pred_v3) <= 0.25) * 100

print(f'MAE (v3, MSE + peso²): {mae_v3:.3f}')
print(f'Acerto exato: {within_exact_v3:.1f}%')
print(f'Tolerância 1 classe: {within_one_class_v3:.1f}%')

test_df_eval_v3 = test_df.copy()
test_df_eval_v3['pred'] = y_pred_v3
test_df_eval_v3['abs_error'] = np.abs(test_df_eval_v3['bcs'] - test_df_eval_v3['pred'])
print(test_df_eval_v3.groupby('bcs')['abs_error'].mean())


MAE (v3, MSE + peso²): 0.211
Acerto exato: 35.2%
Tolerância 1 classe: 66.3%
bcs
3.25    0.294362
3.50    0.176744
3.75    0.179202
4.00    0.189615
4.25    0.299687
Name: abs_error, dtype: float64




| Classe | Original | v2 (Huber + peso linear) | v3 (MSE + peso²) |
|---|---|---|---|
| 3.25 | 0.330 | 0.306 | **0.294** |
| 3.50 | 0.161 | **0.152** | 0.177 |
| 3.75 | **0.141** | 0.154 | 0.179 |
| 4.00 | **0.187** | 0.200 | 0.190 |
| 4.25 | 0.338 | 0.325 | **0.300** |
| MAE geral | 0.207 | **0.206** | 0.211 |
| Tolerância ±0.25 | 66.4% | **67.0%** | 66.3% |


